# S100 every-cycle uniform bulk validation for $L=28,32,36,40$

This notebook runs the uniform topological adaptive circuit with domain walls disabled and perfect correction enabled. For every independent trajectory it evaluates the legacy central tripartition real-space Chern estimator $C_G(t)$ and the global charge at every cycle $t=0,\ldots,40$. Only these compact observables are saved; covariance and occupied-frame histories are never written.

## 1. Runtime setup

Select an **A100 GPU** runtime before continuing. Google Drive is mounted once; executable code is then copied to local `/content` scratch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Complete campaign configuration

These are the preregistered production values. `REPORT_ONLY=True` inspects resume status without starting GPU work. Set `MAX_NEW_TASKS` to a positive integer only when intentionally limiting one Colab session.

In [ ]:
from pathlib import Path

BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/03_uniform_bulk_validation_large_l')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/uniform_perfect_correction_40cycle_s100_l28_l40_batched_v2')
LOCAL_BUNDLE_DIR = Path('/content/03_uniform_bulk_validation_large_l')
SCRATCH_ROOT = Path('/content/uniform_bulk_validation_large_l_scratch')
REPORT_ONLY = False
MAX_NEW_TASKS = None

CONFIG = {
    'sampling_revision': 'uniform_perfect_correction_40cycle_s100_l28_l40_batched_v2',
    'root_seed': 2026090301,
    'sizes': [28, 32, 36, 40],
    'nshell_values': [1, 2, None],
    'samples_per_case': 100,
    'batch_size_by_L': {'28': 100, '32': 50, '36': 30, '40': 20},
    'cycles': 40,
    'device': 'cuda:0',
    'dtype': 'complex128',
    'protocol': {
        'DW': False,
        'alpha_1': 1.0,
        'alpha_2': 1.0,
        'filling_frac': 0.5,
        'trial_orbitals': 'X',
        'init_mode': 'default',
        'sequence': 'random',
        'perfect_correction': True,
        'postselect': False,
        'postselect_probability': 0.0,
        'meas_slab_only': False,
    },
}


## 3. Stage locally and run/resume

The runner verifies existing result/completion pairs, shows one outer batch progress bar, and surfaces the canonical engine's inner cycle bar. Completed batches are skipped; only the batch interrupted in memory is repeated. The size-dependent batches target a largest restart unit of approximately two hours and deliberately do not checkpoint within a batch.

In [ ]:
import codecs
import json
import os
import shutil
import subprocess
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/uniform_bulk_validation_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')

command = [
    sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
    '--config', str(config_path),
    '--output-root', str(OUTPUT_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_TASKS is not None:
    command.extend(['--max-new-tasks', str(int(MAX_NEW_TASKS))])
print('[launch]', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
if process.stdout is None:
    raise RuntimeError('Runner output pipe was not created')
decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
while True:
    raw = os.read(process.stdout.fileno(), 4096)
    if not raw:
        break
    sys.stdout.write(decoder.decode(raw))
    sys.stdout.flush()
tail = decoder.decode(b'', final=True)
if tail:
    sys.stdout.write(tail)
    sys.stdout.flush()
returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
print('[notebook] runner exited successfully', flush=True)


## 4. Release the runtime

Run this final cell only after the runner exits successfully.

In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
